# C9 — Diagnostic de l'arrêt anticipé (famille A, ESC-50)

Dans le calcul principal de C9, le F1 de test de chaque tour suit le nombre d'époques retenu
au réglage. Le taux d'apprentissage suit un cosinus sur 150 époques : une patience de 15 peut
donc arrêter le réglage en plein palier à taux élevé, avant la décroissance.

Ce notebook rejoue **uniquement le réglage** de deux tours, sur les 150 époques complètes
(`patience = None`), avec la graine 42 et le taux d'apprentissage retenu à l'époque :

| Réseau | Fold de test | Fold de validation | lr | Arrêt du calcul principal |
| --- | --- | --- | --- | --- |
| CNN | 5 | 1 | 1e-3 | époque 43, F1 de validation 0,659 |
| CRNN | 2 | 3 | 3e-4 | époque 44, F1 de validation 0,740 |

Le fold de test n'est jamais utilisé. Les log-Mel sont ceux du calcul principal (sortie du
kernel `echolens-c9-family-a-esc50`, attachée en entrée) : ils ne sont pas recalculés (R6).

## a. Démarrage

Même amorçage que les autres notebooks : clone de la branche `BRANCH`, installation éditable,
`src/` ajouté au chemin d'import.

In [ ]:
import sys

BRANCH = "main"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. GPU

In [ ]:
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU : activer l'accélérateur GPU T4 du notebook.")
device = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(device)
print("GPU :", GPU_NAME, "· torch", torch.__version__)
NOTEBOOK_START = time.time()

## c. Log-Mel du calcul principal

On relit `esc50_log_mels.npz` dans les entrées du kernel, et on vérifie sa forme
(2 000 × 64 × 501) et son ordre (celui du manifeste versionné).

In [ ]:
from pathlib import Path

from echolens.core.data.manifest import load_manifest
from echolens.training.family_a import load_log_mels

found = sorted(Path("/kaggle/input").glob("**/esc50_log_mels.npz"))
if not found:
    raise RuntimeError(
        "esc50_log_mels.npz introuvable : attacher la sortie de echolens-c9-family-a-esc50."
    )
log_mel_set = load_log_mels(found[0])
rows = load_manifest(Path(CLONE_DIR) / "data" / "manifests" / "esc50.csv")
print("Log-Mel :", found[0], log_mel_set.log_mels.shape)
if log_mel_set.log_mels.shape != (2000, 64, 501):
    raise RuntimeError(f"Forme inattendue : {log_mel_set.log_mels.shape}")
if log_mel_set.paths != [row.path for row in rows]:
    raise RuntimeError("L'ordre des log-Mel n'est pas celui du manifeste")

## d. Réglage sur 150 époques complètes

Même code que le calcul principal (`train_with_early_stopping`), mais avec `patience = None` :
l'entraînement va jusqu'à 150 époques, et l'on garde le F1 de validation de chaque époque. Le
cosinus est le même (sur 150 époques), la graine aussi (42) : les 43 ou 44 premières époques
refont donc le calcul principal, ce qui sert de contrôle de reproductibilité.

In [ ]:
from echolens.core.data.splits import esc50_round
from echolens.training.family_a import load_family_a_config
from echolens.training.family_a_esc50 import round_indices
from echolens.training.family_a_models import LabeledSpectrograms, train_with_early_stopping

FOLD_COUNT = 5
CLASS_COUNT = 50
SEED = 42
# (réseau, fold de test, taux d'apprentissage, époque et F1 retenus par le calcul principal)
CASES = [
    ("cnn", 5, 1e-3, 43, 0.659),
    ("crnn", 2, 3e-4, 44, 0.740),
]

base_config = load_family_a_config()
no_patience = base_config.training.model_copy(update={"patience": None})
config = base_config.model_copy(update={"training": no_patience})
ORIGINAL_PATIENCE = base_config.training.patience
print("max_epochs =", config.training.max_epochs, "· patience =", config.training.patience)

histories = {}
gpu_seconds = {}
for architecture, test_fold, learning_rate, _, _ in CASES:
    roles = esc50_round(test_fold, FOLD_COUNT)
    indices = round_indices(log_mel_set, roles)
    train = LabeledSpectrograms(
        log_mels=log_mel_set.log_mels[indices.tuning_train],
        labels=log_mel_set.labels[indices.tuning_train],
    )
    validation = LabeledSpectrograms(
        log_mels=log_mel_set.log_mels[indices.validation],
        labels=log_mel_set.labels[indices.validation],
    )
    print(
        f"{architecture} : test {test_fold}, validation {roles.validation_fold}, "
        f"train {roles.tuning_train_folds}, lr {learning_rate:g}"
    )
    start = time.time()
    stopped = train_with_early_stopping(
        architecture, train, validation, CLASS_COUNT, learning_rate, SEED, config, device
    )
    gpu_seconds[architecture] = time.time() - start
    histories[architecture] = stopped.validation_f1_history
    print(f"  {stopped.trained.epochs_run} époques en {gpu_seconds[architecture] / 60:.1f} min")

## e. Courbes, meilleure époque et rejeu de la patience

Pour chaque réseau : le F1 de validation toutes les 10 époques, la meilleure valeur sur les
150 époques et son époque, puis l'époque où une patience de 15 se serait arrêtée sur cette même
courbe (à comparer à 43 et 44 du calcul principal).

In [ ]:
import json

import matplotlib.pyplot as plt


def replay_patience(history: list[float], patience: int) -> tuple[int, float, int]:
    # Meilleure époque, meilleur F1 et époque d'arrêt, avec la règle du calcul principal.
    best_f1 = -1.0
    best_epoch = 0
    for epoch, score in enumerate(history, start=1):
        if score > best_f1:
            best_f1 = score
            best_epoch = epoch
        if epoch - best_epoch >= patience:
            return best_epoch, best_f1, epoch
    return best_epoch, best_f1, len(history)


diagnostic = {"gpu": GPU_NAME, "torch_version": torch.__version__, "seed": SEED, "cases": []}
for architecture, test_fold, learning_rate, original_epoch, original_f1 in CASES:
    history = histories[architecture]
    best_f1 = max(history)
    best_epoch = history.index(best_f1) + 1
    replay_epoch, replay_f1, stop_epoch = replay_patience(history, ORIGINAL_PATIENCE)
    every_ten = {}
    for epoch in range(10, len(history) + 1, 10):
        every_ten[epoch] = round(history[epoch - 1], 4)
    print(f"== {architecture} (test {test_fold}, lr {learning_rate:g})")
    print("  F1 de validation toutes les 10 époques :", every_ten)
    print(f"  meilleur sur 150 époques : {best_f1:.4f} à l'époque {best_epoch}")
    print(
        f"  patience {ORIGINAL_PATIENCE} rejouée : arrêt à l'époque {stop_epoch}, "
        f"meilleure {replay_epoch} ({replay_f1:.4f})"
    )
    print(f"  calcul principal : meilleure {original_epoch} ({original_f1})")
    diagnostic["cases"].append(
        {
            "architecture": architecture,
            "test_fold": test_fold,
            "learning_rate": learning_rate,
            "validation_macro_f1_per_epoch": history,
            "best_epoch": best_epoch,
            "best_validation_macro_f1": best_f1,
            "patience_replay_best_epoch": replay_epoch,
            "patience_replay_best_f1": replay_f1,
            "patience_replay_stop_epoch": stop_epoch,
            "original_best_epoch": original_epoch,
            "original_best_f1": original_f1,
            "gpu_seconds": gpu_seconds[architecture],
        }
    )

figure, axes = plt.subplots(1, len(CASES), figsize=(12, 4))
for column, (architecture, test_fold, learning_rate, original_epoch, _) in enumerate(CASES):
    history = histories[architecture]
    epochs = range(1, len(history) + 1)
    axes[column].plot(epochs, history)
    axes[column].axvline(
        original_epoch, color="red", linestyle="--", label="arrêt du calcul principal"
    )
    axes[column].set_title(
        f"{architecture.upper()}, validation fold {test_fold % 5 + 1}, lr {learning_rate:g}"
    )
    axes[column].set_xlabel("époque")
    axes[column].set_ylabel("F1 macro de validation")
    axes[column].legend()
figure.tight_layout()
figure.savefig("/kaggle/working/c9_early_stopping_diagnostic.png", dpi=150)
plt.show()

total_minutes = (time.time() - NOTEBOOK_START) / 60
diagnostic["notebook_minutes"] = total_minutes
Path("/kaggle/working/c9_early_stopping_diagnostic.json").write_text(
    json.dumps(diagnostic, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Durée totale du notebook (GPU allumé) : {total_minutes:.1f} min")

In [ ]:
import shutil

shutil.rmtree(CLONE_DIR, ignore_errors=True)
print(sorted(path.name for path in Path("/kaggle/working").iterdir()))